<a href="https://colab.research.google.com/github/karishmabanthiacs24/AI_LAB/blob/main/1WN24CS130_week6_PropositionalLogic_TruthTableEnumeration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import re
import pandas as pd

def tokenize(s):
    pattern = r'<->|->|[()&|~]|[A-Za-z][A-Za-z0-9_]*'
    s = s.replace(" ", "")
    tokens = re.findall(pattern, s)
    if "".join(tokens) != s:
        raise ValueError("Invalid expression")
    return tokens

class Parser:
    def __init__(self, tokens):
        self.t = tokens
        self.i = 0

    def match(self, *ops):
        if self.i >= len(self.t):
            return None
        token = self.t[self.i]
        if ops and token not in ops:
            return None
        self.i += 1
        return token

    def parse(self):
        node = self.iff()
        if self.i != len(self.t):
            raise ValueError("Invalid expression")
        return node

    def iff(self):
        x = self.imp()
        while self.match("<->"):
            x = ("<->", x, self.imp())
        return x

    def imp(self):
        x = self.or_expr()
        if self.match("->"):
            return ("->", x, self.imp())
        return x

    def or_expr(self):
        x = self.and_expr()
        while self.match("|"):
            x = ("|", x, self.and_expr())
        return x

    def and_expr(self):
        x = self.neg()
        while self.match("&"):
            x = ("&", x, self.neg())
        return x

    def neg(self):
        if self.match("~"):
            return ("~", self.neg())
        if self.match("("):
            x = self.iff()
            if not self.match(")"):
                raise ValueError("Missing )")
            return x
        token = self.match()
        if token is None or not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]*", token):
            raise ValueError("Expected symbol")
        return ("symbol", token)

def symbols(node):
    if node[0] == "symbol":
        return {node[1]}
    if node[0] == "~":
        return symbols(node[1])
    return symbols(node[1]) | symbols(node[2])

def evaluate(node, model):
    op = node[0]
    if op == "symbol":
        return model[node[1]]
    if op == "~":
        return not evaluate(node[1], model)

    a = evaluate(node[1], model)
    b = evaluate(node[2], model)

    if op == "&":
        return a and b
    if op == "|":
        return a or b
    if op == "->":
        return (not a) or b
    if op == "<->":
        return a == b

def display_truth_table(kb_input, query_input):
    try:
        kb = Parser(tokenize(kb_input)).parse()
        query = Parser(tokenize(query_input)).parse()
        all_symbols = sorted(symbols(kb) | symbols(query))

        def get_models(syms, current_model):
            if not syms:
                return [current_model]
            p = syms[0]
            rest = syms[1:]
            m_true = current_model.copy()
            m_true[p] = True
            m_false = current_model.copy()
            m_false[p] = False
            return get_models(rest, m_true) + get_models(rest, m_false)

        models = get_models(all_symbols, {})

        rows = []
        for m in models:
            kb_val = evaluate(kb, m)
            q_val = evaluate(query, m)
            row = {**m, 'KB': kb_val, 'Query': q_val, 'Entails (KB -> Query)': (not kb_val) or q_val}
            rows.append(row)

        df = pd.DataFrame(rows)
        display(df)

        is_entailed = df['Entails (KB -> Query)'].all()
        if is_entailed:
            print("\nQuery IS ENTAILED by the KB.")
        else:
            print("\nQuery is NOT ENTAILED by the KB.")

    except ValueError as e:
        print("Error:", e)

# Take KB and the number of queries from the user
kb_expr = input("Enter Knowledge Base (KB): ")
print(f"KB entered: {kb_expr}\n")

try:
    n = int(input("Enter the number of queries to check: "))
    print(f"Number of queries: {n}\n")
    for i in range(1, n + 1):
        print(f"--- Checking Query {i} ---")
        query_expr = input(f"Enter Query {i}: ")
        print(f"Query {i}: {query_expr}")
        display_truth_table(kb_expr, query_expr)
        print("\n" + "="*40 + "\n")
except ValueError:
    print("Please enter a valid integer for the number of queries.")

Enter Knowledge Base (KB): (P<->Q)&(Q|R)&(~P->R)
KB entered: (P<->Q)&(Q|R)&(~P->R)

Enter the number of queries to check: 3
Number of queries: 3

--- Checking Query 1 ---
Enter Query 1: Q
Query 1: Q


,P,Q,R,KB,Query,Entails (KB -> Query)
0,True,True,True,True,True,True
1,True,True,False,True,True,True
2,True,False,True,False,False,True
3,True,False,False,False,False,True
4,False,True,True,False,True,True
5,False,True,False,False,True,True
6,False,False,True,True,False,False
7,False,False,False,False,False,True



Query is NOT ENTAILED by the KB.


--- Checking Query 2 ---
Enter Query 2: P->R
Query 2: P->R


,P,Q,R,KB,Query,Entails (KB -> Query)
0,True,True,True,True,True,True
1,True,True,False,True,False,False
2,True,False,True,False,True,True
3,True,False,False,False,False,True
4,False,True,True,False,True,True
5,False,True,False,False,True,True
6,False,False,True,True,True,True
7,False,False,False,False,True,True



Query is NOT ENTAILED by the KB.


--- Checking Query 3 ---
Enter Query 3: R->Q
Query 3: R->Q


,P,Q,R,KB,Query,Entails (KB -> Query)
0,True,True,True,True,True,True
1,True,True,False,True,True,True
2,True,False,True,False,False,True
3,True,False,False,False,True,True
4,False,True,True,False,True,True
5,False,True,False,False,True,True
6,False,False,True,True,False,False
7,False,False,False,False,True,True



Query is NOT ENTAILED by the KB.


